## Preprocessing 

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load the dataset and preserve original_df
original_df = pd.read_excel("Online Retail.xlsx")
print("Shape:", original_df.shape)

Shape: (541909, 8)


In [3]:
# Display the first few rows of the dataset
original_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
# Summarize numeric columns and transaction dates
original_df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


In [5]:
# Inspect column types and non-null counts
original_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [ ]:
# Count missing values and exact duplicates, retain duplicates for now
display(original_df.isna().sum())
print("Number of duplicate rows:", original_df.duplicated().sum())

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Number of duplicate rows: 5268


In [7]:
# Keep transactions with a known CustomerID for customer-level analysis
customer_df = original_df.dropna(subset=['CustomerID'])
print("New Shape:", customer_df.shape)

New Shape: (406829, 8)


In [8]:
# Identify cancellation rows among transactions with a known CustomerID
canceled_transactions = customer_df[customer_df["InvoiceNo"].astype(str).str.startswith("C")]

print("Canceled rows:", len(canceled_transactions))
print("Unique canceled invoices:", canceled_transactions["InvoiceNo"].nunique())
print("Customers with cancellations:", canceled_transactions["CustomerID"].nunique())

Canceled rows: 8905
Unique canceled invoices: 3654
Customers with cancellations: 1589


In [9]:
# Exclude cancellation rows, original purchases remain in the data
non_canceled_df = customer_df[~customer_df["InvoiceNo"].astype(str).str.startswith("C")]
non_canceled_df.shape

(397924, 8)

In [10]:
# Select rows with zero or negative quantities or prices for analysis
non_positive_rows = non_canceled_df[(non_canceled_df["Quantity"] <= 0) | (non_canceled_df["UnitPrice"] <= 0)]

In [11]:
# Keep positive quantities and prices, then copy for further preprocessing
positive_sales_df = non_canceled_df[(non_canceled_df["Quantity"] > 0) & (non_canceled_df["UnitPrice"] > 0)]
cleaned_df = positive_sales_df.copy()

In [12]:
# Calculate spending per line item before refunds and cancellations
cleaned_df["TotalPrice"] = cleaned_df["Quantity"] * cleaned_df["UnitPrice"]

In [13]:
# Report cleaned purchase rows, total rows removed, and unique customers
print("\nCleaned rows:", len(cleaned_df))
print("Rows removed:", len(original_df) - len(cleaned_df))
print("Unique customers:", cleaned_df["CustomerID"].nunique())


Cleaned rows: 397884
Rows removed: 144025
Unique customers: 4338


In [14]:
display(cleaned_df.head())

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
